# Map EV Charging Stations on Highway I-5

Suppose you own an electric vehicle (EV).  You want to drive one of the most important trade routes in the western USA, the I-5.  I-5 is 1,381 miles of highway from southern California to northern Washington.  Truckers deliver goods from Mexico to Canada with stops at distribution centers and ports in Los Angeles, San Francisco, Portland, Seattle, and other cities.  Families drive to vacation spots in Yosemite National Forest and other popular destinations.  Where will you stop and charge your EV?  How many EV charging stations are there along I-5?  This notebook starts exploring data about EV charging stations published by the Alternative Fuels Data Center of the USA Department of Energy.

First, is this data current?  According to the USA Department of Energy, data from 2049 stations was confirmed in 2019.  Data from 99 stations was confirmed between March and November 2018.

Next, what does the data tell us about EV charging stations along I5?
- 606 stations offer FREE refueling

- 1938 stations offer the J1772 EV connector type.  187 stations offer TESLA connector types and 25 stations have only CHADEMO.

- EV Networks include:  Blink Network, ChargePoint Network, EV Connect, eVgo Network, GE, Greenlots, OpConnect, SemaCharge Network, Tesla Destination, Tesla,Electrify America, WattStation, and Webasto

- Unfortunately, 2134 stations are missing a list of cards that they accept so drivers cannot verifiy that their card will be accepted

The following code was used to compute these statistics.  Comments are included explaining the logic.  

In [1]:
# import Pandas to process the data and Folium for drawing maps

import pandas as pd
import folium
from folium.plugins import MarkerCluster

pd.set_option('display.max_columns',23)

d = pd.read_csv("/kaggle/input/doe-ev-charging-stations-i5/stations.csv")

# select data to explore
df = d[["Station Name","Street Address","City","State","ZIP",
        "Geocode Status","Latitude","Longitude",
        "Date Last Confirmed","Groups With Access Code","Access Days Time",
        "Cards Accepted","Access Code","Access Detail Code",
        "EV Level1 EVSE Num","EV Level2 EVSE Num","EV DC Fast Count",
        "EV Network","EV Pricing","EV Connector Types"]]

In [2]:
# get a list of the EV charging stations in California along with location and GPS coordinates

Location = df["Station Name"].loc[df['State'] =='CA'].tolist()
Latitude = df["Latitude"].loc[df['State'] =='CA'].tolist()
Longitude = df["Longitude"].loc[df['State'] =='CA'].tolist()
print("Number of Stations in California ", len(Location))

# create a Pandas dataframe and load these lists

sm = pd.DataFrame(columns = {'Location','Latitude','Longitude'})
sm.columns = ['Location','Latitude','Longitude']
sm['Location']  = Location
sm['Latitude']  = Latitude
sm['Longitude'] = Longitude

# use the dataframe to draw the map
# center the map on the mean of latitude and longitude 
# use the OpenStreetMap map format and set map view using zoom_start
map_world = folium.Map(location=[sm.Latitude.mean(), sm.Longitude.mean()], tiles = 'OpenStreetMap', zoom_start = 6)

#  add Locations to map
for lat, lng, label in zip(sm.Latitude, sm.Longitude, sm.Location):
    folium.CircleMarker(
        [lat, lng],
        radius=1,
        popup=label,
        fill=True,
        color='Blue',
        fill_color='Yellow',
        fill_opacity=0.6
        ).add_to(map_world)

map_world

Number of Stations in California  1373


In [3]:
# gather the location of stations in Oregon

Location = df["Station Name"].loc[df['State'] =='OR'].tolist()
Latitude = df["Latitude"].loc[df['State'] =='OR'].tolist()
Longitude = df["Longitude"].loc[df['State'] =='OR'].tolist()
print("Number of Stations in Oregon ", len(Location))

sm = pd.DataFrame(columns = {'Location','Latitude','Longitude'})
sm.columns = ['Location','Latitude','Longitude']
sm['Location']  = Location
sm['Latitude']  = Latitude
sm['Longitude'] = Longitude

map_world = folium.Map(location=[sm.Latitude.mean(), sm.Longitude.mean()], tiles = 'OpenStreetMap', zoom_start = 6)

#  map the locations in Oregon
for lat, lng, label in zip(sm.Latitude, sm.Longitude, sm.Location):
    folium.CircleMarker(
        [lat, lng],
        radius=1,
        popup=label,
        fill=True,
        color='Blue',
        fill_color='Yellow',
        fill_opacity=0.6
        ).add_to(map_world)

map_world

Number of Stations in Oregon  311


In [4]:
# gather the location of stations in Washington

Location = df["Station Name"].loc[df['State'] =='WA'].tolist()
Latitude = df["Latitude"].loc[df['State'] =='WA'].tolist()
Longitude = df["Longitude"].loc[df['State'] =='WA'].tolist()
print("Number of Stations in Washington ", len(Location))

sm = pd.DataFrame(columns = {'Location','Latitude','Longitude'})
sm.columns = ['Location','Latitude','Longitude']
sm['Location']  = Location
sm['Latitude']  = Latitude
sm['Longitude'] = Longitude

map_world = folium.Map(location=[sm.Latitude.mean(), sm.Longitude.mean()], tiles = 'OpenStreetMap', zoom_start = 7)

#  map the locations in Washington
for lat, lng, label in zip(sm.Latitude, sm.Longitude, sm.Location):
    folium.CircleMarker(
        [lat, lng],
        radius=1,
        popup=label,
        fill=True,
        color='Blue',
        fill_color='Yellow',
        fill_opacity=0.6
        ).add_to(map_world)

map_world

Number of Stations in Washington  464


In [5]:
print("\nUNIQUE CLASSIFIERS")

# fill any missing data with the word "missing"
df = df.fillna('missing')

print("Geocode Status")
print(df["Geocode Status"].unique())

# The information about geocode status is from the Colorado data marketplace:
# https://data.colorado.gov/Energy/Alternative-Fuels-and-Electric-Vehicle-Charging-St/team-3ugz
# It is a rating indicating the approximate accuracy of the latitude and longitude for the station's address, 
# given as code values: 
#    GPS = The location is from a real GPS readout at the station. 
#    200-9 = Premise (building name, property name, shopping center, etc.) level accuracy. 
#    200-8 = Address level accuracy. 
#    200-7 = Intersection level accuracy. 
#    200-6 = Street level accuracy. 
#    200-5 = ZIP code (postal code) level accuracy. 
#    200-4 = Town (city, village) level accuracy. 
#    200-3 = Sub-region (county, municipality, etc.) level accuracy. 
#    200-2 = Region (state, province, prefecture, etc.) level accuracy. 
#    200-1 = Country level accuracy. 
#    200-0 = Unknown accuracy.
# This data could be used to finetune the script to add a note about the accuracy of the location



UNIQUE CLASSIFIERS
Geocode Status
['GPS' '200-9' '200-8']


In [6]:
# in order to charge you must have the correct EV connector type for your electric vehicle

print("\nEV Connector Types\n")
print(df["EV Connector Types"].unique())
numJ1772 = df["EV Connector Types"].str.contains('J1772').value_counts().tolist()
print("\n",numJ1772[0]," stations have EV connector type:  J1772\n")
EVcon = df["EV Connector Types"].value_counts().copy()
for index, val in EVcon.iteritems():
    print(val, " stations have ", index)


EV Connector Types

['J1772' 'CHADEMO J1772COMBO' 'TESLA' 'J1772 J1772COMBO'
 'CHADEMO J1772 J1772COMBO' 'CHADEMO' 'J1772 NEMA520' 'CHADEMO J1772'
 'J1772COMBO' 'CHADEMO J1772COMBO NEMA520' 'J1772 NEMA1450'
 'J1772 NEMA515']

 1938  stations have EV connector type:  J1772

1641  stations have  J1772
187  stations have  TESLA
114  stations have  CHADEMO J1772COMBO
76  stations have  CHADEMO J1772 J1772COMBO
47  stations have  CHADEMO J1772
26  stations have  J1772COMBO
25  stations have  J1772 NEMA520
25  stations have  CHADEMO
4  stations have  J1772 J1772COMBO
2  stations have  J1772 NEMA515
2  stations have  J1772 NEMA1450
1  stations have  CHADEMO J1772COMBO NEMA520


In [7]:
# in order to charge you will need to use your EV network along with the proper access codes

print("\nAccess Code")
print(df["Access Code"].unique())

print("\nGroups With Access Code")
print(df["Groups With Access Code"].unique())

print("\nAccess Detail Code")
print(df["Access Detail Code"].unique())

print("\nEV Network")
print(df["EV Network"].unique())

print("\nCards Accepted")
print(df["Cards Accepted"].unique())


Access Code
['public']

Groups With Access Code
['Public' 'Public - Card key at all times' 'Public - Call ahead'
 'Public - Credit card at all times']

Access Detail Code
['missing' 'KEY_ALWAYS' 'CALL' 'CREDIT_CARD_ALWAYS']

EV Network
['Non-Networked' 'ChargePoint Network' 'OpConnect' 'Blink Network'
 'Greenlots' 'Tesla Destination' 'EV Connect' 'eVgo Network'
 'SemaCharge Network' 'Tesla' 'Electrify America' 'GE WattStation'
 'Webasto']

Cards Accepted
['missing' 'A D Debit M V' 'Cash M V' 'A Cash Checks D M V' 'A M V' 'M V'
 'A D M V' 'D M V']


In [8]:
# many stations offer free charging and others charge a fee

print("\nEV Pricing")
df['EV Pricing'] = df['EV Pricing'].str.upper()
EVprice = df["EV Pricing"].value_counts().copy()
for index, val in EVprice.head(5).iteritems():
    print(val, " stations have pricing set at ", index)


EV Pricing
606  stations have pricing set at  FREE
197  stations have pricing set at  MISSING
128  stations have pricing set at  LEVEL 2: $0.49 PER KWH
77  stations have pricing set at  PRICING IS BASED ON LENGTH OF TIME. $1.25 PER HOUR FOR THE FIRST 4 HOURS, AND $3 PER ADDITIONAL HOUR, MAXIMUM PAYMENT OF $24, MAXIMUM SESSION TIME IS 24 HOURS
70  stations have pricing set at  LEVEL 2: $2.00 PER HOUR, MAXIMUM PAYMENT OF $40.00


In [9]:
# Before taking a trip, it is important to know whether the data about charging stations is current and accurate

print("\nDATA CONFIRMATION")
old = df.loc[~df["Date Last Confirmed"].str.contains('2019')].copy()
old.sort_values(by=["Date Last Confirmed"],inplace=True)
print(old.shape[0], " stations were confirmed BEFORE 2019 between ", end="")
print(old['Date Last Confirmed'].iloc[0], " and ", end="")
print(old['Date Last Confirmed'].iloc[-1])


DATA CONFIRMATION
101  stations were confirmed BEFORE 2019 between 2018-03-08  and missing


In [10]:
print("\nMISSING CARD DATA")
#print(df["Cards Accepted"].isnull().sum())
print(df["Cards Accepted"].str.contains('missing').sum(), " stations are missing the cards they accept")


MISSING CARD DATA
2134  stations are missing the cards they accept


In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.

/kaggle/input/doe-ev-charging-stations-i5/stations.csv
/kaggle/input/doe-ev-charging-stations-i5/iotaElaadnl.png
